# 01 — Exploratory Data Analysis (EDA)
## Airline Flight Delay Prediction Project

---

### Purpose of this notebook

Before we build any model, we need to **deeply understand the data**. This notebook answers:

1. **What does our dataset look like?** — shape, types, missing values
2. **What is our target variable?** — distribution, range, balance
3. **Univariate analysis** — how is each feature distributed?
4. **Bivariate analysis** — how do features relate to the target?
5. **Temporal analysis** — how has delay rate changed over 20 years?
6. **Carrier analysis** — which airlines delay the most?
7. **Airport analysis** — which airports have the worst delays?
8. **Seasonality analysis** — which months are worst?
9. **Correlation analysis** — are features redundant?
10. **Key findings** — what will drive our feature engineering?

---

> **Data grain:** Each row is **one airline × one airport × one month × one year** (aggregated).  
> This is NOT a record per individual flight.

---
*Notebook: `01_EDA.ipynb` | Project: Airline Delay DNN | Author: Your Name*

---
## 1. Imports & Configuration

We import all libraries upfront so any missing dependency is caught immediately.  
We also set global plot styles here so every figure in this notebook looks consistent.

| Library | Why |
|---|---|
| `pandas` | DataFrame manipulation |
| `numpy` | Numerical operations |
| `matplotlib` | Base plotting engine |
| `seaborn` | Statistical visualisations |
| `scipy.stats` | Statistical tests (skewness, normality) |
| `warnings` | Suppress non-critical warnings |

In [ ]:
# ── Standard library ──────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')   # keep output clean

# ── Data manipulation ─────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# ── Statistics ────────────────────────────────────────────────
from scipy import stats

# ── Display settings ──────────────────────────────────────────
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Global plot style ─────────────────────────────────────────
# Using a clean, publication-ready style
plt.rcParams.update({
    'figure.dpi'       : 120,
    'figure.facecolor' : 'white',
    'axes.facecolor'   : 'white',
    'axes.grid'        : True,
    'grid.alpha'       : 0.3,
    'font.size'        : 11,
    'axes.titlesize'   : 13,
    'axes.titleweight' : 'bold',
    'axes.labelsize'   : 11,
})
PALETTE  = 'viridis'       # default colour palette
FIG_PATH = '../reports/figures/'   # where all plots will be saved

print("✅ Imports successful")

---
## 2. Load Data

We load the raw CSV and immediately run a **schema validation** — checking that:
- Expected columns are present
- Data types are sensible
- No completely empty columns exist

This is a professional habit: **never assume the file is clean just because it loaded**.

In [ ]:
# ── Load raw CSV ──────────────────────────────────────────────
DATA_PATH = '../data/raw/Airline_Delay_Cause.csv'

df = pd.read_csv(DATA_PATH)

# ── Schema validation ─────────────────────────────────────────
EXPECTED_COLUMNS = [
    'year', 'month', 'carrier', 'carrier_name', 'airport', 'airport_name',
    'arr_flights', 'arr_del15', 'carrier_ct', 'weather_ct', 'nas_ct',
    'security_ct', 'late_aircraft_ct', 'arr_cancelled', 'arr_diverted',
    'arr_delay', 'carrier_delay', 'weather_delay', 'nas_delay',
    'security_delay', 'late_aircraft_delay'
]

missing_cols = set(EXPECTED_COLUMNS) - set(df.columns)
assert len(missing_cols) == 0, f"Missing columns: {missing_cols}"

print(f"✅ Schema validated — all {len(EXPECTED_COLUMNS)} expected columns present")
print(f"\n📊 Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"📅 Year range   : {df['year'].min()} → {df['year'].max()}")
print(f"✈️  Carriers      : {df['carrier'].nunique()} unique")
print(f"🛬  Airports      : {df['airport'].nunique()} unique")

---
## 3. First Look at the Data

### 3.1 Sample rows

Let's see what the data actually looks like. We display the first few rows with all columns visible.

In [ ]:
# ── Display first 5 rows ──────────────────────────────────────
# .T transposes for easier reading when there are many columns
df.head(5)

### 3.2 Data types & memory usage

Understanding dtypes helps us:
- Identify columns that need type conversion
- Estimate memory footprint
- Spot potential encoding issues

In [ ]:
# ── Data types summary ────────────────────────────────────────
dtype_df = pd.DataFrame({
    'dtype'       : df.dtypes,
    'non_null'    : df.notna().sum(),
    'null_count'  : df.isna().sum(),
    'null_pct'    : (df.isna().mean() * 100).round(2),
    'nunique'     : df.nunique(),
    'sample_val'  : [df[c].dropna().iloc[0] if df[c].notna().any() else 'ALL NULL'
                     for c in df.columns]
})

# Highlight rows with missing values
dtype_df.style.background_gradient(
    subset=['null_pct'], cmap='Reds', vmin=0, vmax=1
)

### 3.3 Statistical summary

`describe()` gives us a quick view of the range, central tendency, and spread of every numeric column.

**Things to look for:**
- Min values of 0 where negative doesn't make sense ✅
- Max values that seem unreasonably large (potential outliers) ⚠️
- Std >> Mean (high variance columns)

In [ ]:
# ── Statistical summary of numeric columns ────────────────────
df.describe().T.style.background_gradient(cmap='Blues', subset=['mean', 'std'])

---
## 4. Missing Values Analysis

~490 rows have missing values across the numeric columns. Before we decide how to handle them,  
we need to understand **why they are missing** — is it random, or is there a pattern?

**Hypothesis:** Missing rows may correspond to carrier-airport combinations that didn't  
operate in a particular month — a structural absence, not a data error.

In [ ]:
# ── Overall missing value counts ─────────────────────────────
missing = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(3)

missing_summary = pd.DataFrame({
    'missing_count' : missing[missing > 0],
    'missing_pct'   : missing_pct[missing > 0]
}).sort_values('missing_count', ascending=False)

print("Columns with missing values:")
print(missing_summary.to_string())
print(f"\nTotal rows with ANY null: {df.isnull().any(axis=1).sum():,}")
print(f"That is {df.isnull().any(axis=1).mean()*100:.2f}% of all rows")

In [ ]:
# ── Visualise: Missing value heatmap ─────────────────────────
# We sample 5000 rows to keep the plot readable
# White = present, coloured = missing
sample_idx = df.sample(5000, random_state=42).index

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(
    df.loc[sample_idx].isnull(),
    cbar=False,
    cmap='viridis',
    yticklabels=False,
    ax=ax
)
ax.set_title('Missing value pattern (5,000-row sample)\nBlack = present, Yellow = missing')
ax.set_xlabel('Columns')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(FIG_PATH + '01_missing_values_heatmap.png', bbox_inches='tight')
plt.show()
print("💾 Saved: 01_missing_values_heatmap.png")

In [ ]:
# ── Are missing rows concentrated in specific years? ──────────
null_rows = df[df.isnull().any(axis=1)]
print("Missing rows by year:")
print(null_rows['year'].value_counts().sort_index().to_string())

# Check if missing numeric rows have carrier/airport info
print(f"\nOf {len(null_rows)} rows with nulls:")
print(f"  - Null carrier     : {null_rows['carrier'].isna().sum()}")
print(f"  - Null airport     : {null_rows['airport'].isna().sum()}")
print(f"  - Null arr_flights : {null_rows['arr_flights'].isna().sum()}")

**Conclusion from missing value analysis:**

- Missing values affect < 0.2% of rows — a very small fraction
- They appear to be structural gaps (routes that didn't operate that month)
- Our strategy: **drop these rows** during preprocessing — they won't meaningfully affect the model
- We will document this decision in `03_preprocessing.ipynb`

---
## 5. Target Variable Engineering & Analysis

Our target does not exist in the raw data — we need to **derive it**.

### Why `delay_rate` and not `arr_del15` directly?

| Option | Problem |
|---|---|
| `arr_del15` (raw count) | Biased by airport size — JFK will always have huge counts |
| `arr_delay` (total minutes) | Aggregated minutes, not useful for prediction |
| `delay_rate = arr_del15 / arr_flights` | ✅ Normalised, comparable across airports, range [0,1] |

> **Definition:** `delay_rate` = proportion of arriving flights that were delayed ≥ 15 minutes.

In [ ]:
# ── Create target variable ────────────────────────────────────
# Drop rows where we can't compute the rate (missing denominator)
df_clean = df.dropna(subset=['arr_flights', 'arr_del15']).copy()

# Exclude rows with zero flights (division by zero / no information)
df_clean = df_clean[df_clean['arr_flights'] > 0].copy()

# Compute delay rate: proportion of flights delayed >= 15 min
df_clean['delay_rate'] = df_clean['arr_del15'] / df_clean['arr_flights']

# Sanity check: rate must be in [0, 1]
assert df_clean['delay_rate'].between(0, 1).all(), "ERROR: delay_rate outside [0,1]!"

print(f"Rows after cleaning: {len(df_clean):,}")
print(f"\n📈 delay_rate statistics:")
print(df_clean['delay_rate'].describe().to_string())

In [ ]:
# ── Visualise target distribution ────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# ---- Plot 1: Histogram with KDE ----
axes[0].hist(df_clean['delay_rate'], bins=60, color='steelblue',
             edgecolor='white', alpha=0.85, density=True)
df_clean['delay_rate'].plot.kde(ax=axes[0], color='darkred', lw=2)
axes[0].axvline(df_clean['delay_rate'].mean(),   color='orange', ls='--', lw=1.5, label=f"Mean={df_clean['delay_rate'].mean():.3f}")
axes[0].axvline(df_clean['delay_rate'].median(), color='green',  ls='--', lw=1.5, label=f"Median={df_clean['delay_rate'].median():.3f}")
axes[0].set_xlabel('delay_rate')
axes[0].set_ylabel('Density')
axes[0].set_title('Distribution of delay_rate')
axes[0].legend()

# ---- Plot 2: Box plot (detect outliers) ----
axes[1].boxplot(df_clean['delay_rate'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6),
                medianprops=dict(color='darkred', lw=2))
axes[1].set_title('Box plot — outlier detection')
axes[1].set_ylabel('delay_rate')
axes[1].set_xlabel('')

# ---- Plot 3: Q-Q plot (check normality) ----
stats.probplot(df_clean['delay_rate'], dist='norm', plot=axes[2])
axes[2].set_title('Q-Q plot (Normal distribution)')

fig.suptitle('Target Variable: delay_rate = arr_del15 / arr_flights', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIG_PATH + '02_target_distribution.png', bbox_inches='tight')
plt.show()
print("💾 Saved: 02_target_distribution.png")

In [ ]:
# ── Skewness and Kurtosis ─────────────────────────────────────
# These are important because they tell us if we need to transform the target
skewness = df_clean['delay_rate'].skew()
kurtosis = df_clean['delay_rate'].kurtosis()

print(f"Skewness : {skewness:.4f}")
print(f"Kurtosis : {kurtosis:.4f}")
print()
if abs(skewness) < 0.5:
    print("✅ Skewness is low — distribution is approximately symmetric")
elif abs(skewness) < 1.0:
    print("⚠️  Moderate skewness — consider log or sqrt transform during preprocessing")
else:
    print("🚨 High skewness — strongly recommend a transformation")

# ── Shapiro-Wilk normality test on a sample ───────────────────
# (Full dataset is too large for Shapiro-Wilk, use 5000-sample)
sample = df_clean['delay_rate'].sample(5000, random_state=42)
stat, p_value = stats.shapiro(sample)
print(f"\nShapiro-Wilk test (n=5000): stat={stat:.4f}, p={p_value:.4e}")
if p_value < 0.05:
    print("❌ Target is NOT normally distributed (p < 0.05)")
    print("   → We will use MAE as the primary metric (robust to non-normality)")

---
## 6. Univariate Analysis

We examine each **input feature** individually — understanding the distribution of a feature  
before modelling is essential. We are looking for:

- Heavily skewed features that may need transformation
- Bimodal distributions that hint at subgroups
- Features with extreme outliers
- Categorical cardinality (number of unique values)

We split features into groups:
1. **Numeric flight volume features** — `arr_flights`, `arr_cancelled`, `arr_diverted`
2. **Temporal features** — `year`, `month`
3. **Categorical features** — `carrier`, `airport`

> **Note:** We deliberately skip the leaky delay-cause columns (`carrier_ct`, `weather_ct`, etc.)  
> since they will not be used as model inputs.

In [ ]:
# ── 6.1 Numeric flight volume features ──────────────────────
# These are the safe (non-leaky) numeric input features
numeric_features = ['arr_flights', 'arr_cancelled', 'arr_diverted']

fig, axes = plt.subplots(3, 3, figsize=(16, 12))

for i, col in enumerate(numeric_features):
    data = df_clean[col].dropna()

    # Row 0: Histogram + KDE
    axes[i, 0].hist(data, bins=50, color='steelblue', edgecolor='white', alpha=0.8, density=True)
    data.plot.kde(ax=axes[i, 0], color='darkred', lw=2)
    axes[i, 0].set_title(f'{col} — Distribution')
    axes[i, 0].set_xlabel(col)

    # Row 1: Box plot
    axes[i, 1].boxplot(data, patch_artist=True,
                       boxprops=dict(facecolor='steelblue', alpha=0.6),
                       medianprops=dict(color='darkred', lw=2),
                       flierprops=dict(marker='o', markersize=1, alpha=0.3))
    axes[i, 1].set_title(f'{col} — Outliers')

    # Row 2: Log-transformed distribution (handle zero with log1p)
    log_data = np.log1p(data)
    axes[i, 2].hist(log_data, bins=50, color='teal', edgecolor='white', alpha=0.8)
    axes[i, 2].set_title(f'log1p({col})')
    axes[i, 2].set_xlabel(f'log1p({col})')

    skew_val = data.skew()
    log_skew = log_data.skew()
    print(f"{col:20s}  skew={skew_val:6.2f}  log_skew={log_skew:6.2f}  "
          f"max={data.max():,.0f}")

plt.suptitle('Univariate Analysis — Numeric Input Features', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIG_PATH + '03_univariate_numeric.png', bbox_inches='tight')
plt.show()
print("\n💾 Saved: 03_univariate_numeric.png")

### 6.2 Temporal features

`year` and `month` are actually **categorical in nature** — there is no meaningful linear  
relationship between year=2010 and delay rate vs year=2011.

However, they encode:
- **Trend** over time (year)
- **Seasonality** (month) — summer vs winter flying patterns

We will encode month as **cyclic features** (sin/cos) in feature engineering.

In [ ]:
# ── 6.2 Year and month distributions ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Year: count of records per year
year_counts = df_clean['year'].value_counts().sort_index()
axes[0].bar(year_counts.index, year_counts.values, color='steelblue', edgecolor='white')
axes[0].set_title('Records per year')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Number of records')
axes[0].tick_params(axis='x', rotation=45)

# Month: count of records per month
month_counts = df_clean['month'].value_counts().sort_index()
month_names = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']
axes[1].bar(month_names, month_counts.values, color='teal', edgecolor='white')
axes[1].set_title('Records per month')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Number of records')

plt.suptitle('Temporal Feature Coverage', fontsize=14)
plt.tight_layout()
plt.savefig(FIG_PATH + '04_temporal_coverage.png', bbox_inches='tight')
plt.show()
print("💾 Saved: 04_temporal_coverage.png")

In [ ]:
# ── 6.3 Categorical cardinality ──────────────────────────────
# High-cardinality categoricals need special treatment
categorical_cols = ['carrier', 'airport', 'carrier_name']

for col in categorical_cols:
    n_unique = df_clean[col].nunique()
    top5      = df_clean[col].value_counts().head(5)
    print(f"{'─'*50}")
    print(f"  {col}: {n_unique} unique values")
    print(f"  Top 5:")
    for val, cnt in top5.items():
        pct = cnt / len(df_clean) * 100
        print(f"    {val:35s}  {cnt:6,}  ({pct:.1f}%)")

---
## 7. Temporal Analysis — 20 Years of Delay Trends

This is one of the most important analyses in the entire project.  
Understanding **how delay rates changed over time** will:

1. Justify our temporal train/test split strategy
2. Reveal the impact of major events (9/11 aftermath, COVID-19, 2007–08 financial crisis)
3. Show strong seasonality that we need to encode as features

In [ ]:
# ── 7.1 Annual delay rate trend ──────────────────────────────
# We compute the average delay_rate across all carrier-airport pairs per year
# This is the macro-level view of the entire industry

yearly = (
    df_clean
    .groupby('year')['delay_rate']
    .agg(['mean', 'median', 'std'])
    .reset_index()
)

fig, ax = plt.subplots(figsize=(14, 5))

# Shaded confidence band (mean ± std)
ax.fill_between(yearly['year'],
                yearly['mean'] - yearly['std'],
                yearly['mean'] + yearly['std'],
                alpha=0.15, color='steelblue', label='±1 std dev')

ax.plot(yearly['year'], yearly['mean'],   'o-', color='steelblue', lw=2.5,
        label='Mean delay rate')
ax.plot(yearly['year'], yearly['median'], 's--', color='orange', lw=1.5,
        label='Median delay rate')

# ── Annotate major events ──────────────────────────────────────
events = {
    2007: ('Peak delays\n(pre-crisis)', 'red'),
    2020: ('COVID-19\ncollapse', 'purple'),
}
for yr, (label, color) in events.items():
    val = yearly.loc[yearly['year'] == yr, 'mean'].values
    if len(val):
        ax.annotate(label, xy=(yr, val[0]),
                    xytext=(yr+0.5, val[0]+0.03),
                    fontsize=9, color=color,
                    arrowprops=dict(arrowstyle='->', color=color, lw=1.2))

ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
ax.set_xlabel('Year')
ax.set_ylabel('Delay Rate')
ax.set_title('US Airline Delay Rate — 20-Year Trend (2003–2022)')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_PATH + '05_yearly_trend.png', bbox_inches='tight')
plt.show()
print("💾 Saved: 05_yearly_trend.png")

In [ ]:
# ── 7.2 Monthly seasonality ──────────────────────────────────
# Compute mean delay rate per month across all years
monthly = (
    df_clean
    .groupby('month')['delay_rate']
    .agg(['mean', 'std'])
    .reset_index()
)
month_names = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']
monthly['month_name'] = month_names

fig, ax = plt.subplots(figsize=(12, 4))

colors = plt.cm.RdYlGn_r(monthly['mean'] / monthly['mean'].max())
bars = ax.bar(monthly['month_name'], monthly['mean'],
              color=colors, edgecolor='white', width=0.7)
ax.errorbar(monthly['month_name'], monthly['mean'],
            yerr=monthly['std'], fmt='none',
            color='gray', capsize=4, lw=1.2, alpha=0.6)

ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
ax.set_xlabel('Month')
ax.set_ylabel('Mean delay rate')
ax.set_title('Seasonality — Mean Delay Rate by Month\n(error bars = ±1 std)')

# Annotate worst months
for i, row in monthly.iterrows():
    ax.text(i, row['mean'] + 0.003, f"{row['mean']:.1%}",
            ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(FIG_PATH + '06_monthly_seasonality.png', bbox_inches='tight')
plt.show()
print("💾 Saved: 06_monthly_seasonality.png")

In [ ]:
# ── 7.3 Heatmap: Year × Month delay rate ─────────────────────
# This is the most informative single visualisation in the notebook
# We see both the long-term trend AND the seasonal pattern simultaneously

pivot_ym = (
    df_clean
    .groupby(['year', 'month'])['delay_rate']
    .mean()
    .unstack(level='month')
)
pivot_ym.columns = month_names

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    pivot_ym,
    cmap='RdYlGn_r',
    annot=True,
    fmt='.2f',
    linewidths=0.3,
    linecolor='white',
    cbar_kws={'label': 'Mean delay rate', 'format': '%.2f'},
    ax=ax,
    vmin=0.10, vmax=0.35    # fix scale so all years are comparable
)
ax.set_title('Delay Rate Heatmap: Year × Month\n(darker red = worse delays)', fontsize=14)
ax.set_xlabel('Month')
ax.set_ylabel('Year')
plt.tight_layout()
plt.savefig(FIG_PATH + '07_year_month_heatmap.png', bbox_inches='tight')
plt.show()
print("💾 Saved: 07_year_month_heatmap.png")

---
## 8. Carrier Analysis — Which Airlines Delay the Most?

Understanding per-carrier delay behaviour tells us two things:
1. **`carrier` is likely a strong predictive feature** — some airlines are consistently worse
2. We can engineer a **historical carrier delay rate** feature for the model

In [ ]:
# ── 8.1 Carrier delay rate ranking ───────────────────────────
# Compute mean delay rate and volume per carrier
carrier_stats = (
    df_clean
    .groupby('carrier_name')
    .agg(
        mean_delay_rate  = ('delay_rate', 'mean'),
        median_delay_rate= ('delay_rate', 'median'),
        std_delay_rate   = ('delay_rate', 'std'),
        total_flights    = ('arr_flights', 'sum'),
        n_records        = ('delay_rate', 'count')
    )
    .sort_values('mean_delay_rate', ascending=False)
    .reset_index()
)

# Shorten long carrier names for plotting
carrier_stats['short_name'] = carrier_stats['carrier_name'].str.split(' ').str[:2].str.join(' ')

print("Top 10 worst carriers by mean delay rate:")
print(carrier_stats[['carrier_name','mean_delay_rate','total_flights','n_records']].head(10).to_string(index=False))

In [ ]:
# ── 8.2 Carrier ranking bar chart ────────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))

colors = plt.cm.RdYlGn_r(
    np.linspace(0.1, 0.9, len(carrier_stats))
)
bars = ax.barh(carrier_stats['short_name'], carrier_stats['mean_delay_rate'],
               color=colors, edgecolor='white', height=0.7)
ax.errorbar(
    carrier_stats['mean_delay_rate'],
    carrier_stats['short_name'],
    xerr=carrier_stats['std_delay_rate'],
    fmt='none', color='gray', capsize=3, lw=1, alpha=0.5
)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
ax.set_xlabel('Mean delay rate')
ax.set_title('Carrier Delay Rate Ranking (2003–2022)\n(error bars = ±1 std, all months & airports pooled)')
ax.invert_yaxis()   # best carrier at top

# Grand mean reference line
grand_mean = df_clean['delay_rate'].mean()
ax.axvline(grand_mean, color='black', ls='--', lw=1.2, label=f'Grand mean = {grand_mean:.1%}')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_PATH + '08_carrier_ranking.png', bbox_inches='tight')
plt.show()
print("💾 Saved: 08_carrier_ranking.png")

In [ ]:
# ── 8.3 Carrier delay rate over time (top 6 carriers) ────────
# Select top 6 carriers by total flight volume (most data = most reliable trend)
top6_carriers = (
    df_clean
    .groupby('carrier_name')['arr_flights']
    .sum()
    .nlargest(6)
    .index.tolist()
)

carrier_yearly = (
    df_clean[df_clean['carrier_name'].isin(top6_carriers)]
    .groupby(['year', 'carrier_name'])['delay_rate']
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(14, 5))
palette = sns.color_palette('tab10', n_colors=6)
for i, carrier in enumerate(top6_carriers):
    subset = carrier_yearly[carrier_yearly['carrier_name'] == carrier]
    short  = carrier.split(' ')[0]
    ax.plot(subset['year'], subset['delay_rate'], 'o-',
            color=palette[i], lw=2, label=short, markersize=4)

ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
ax.set_xlabel('Year')
ax.set_ylabel('Mean delay rate')
ax.set_title('Delay Rate Trend by Carrier — Top 6 Airlines (by volume)')
ax.legend(ncol=3, fontsize=9)
plt.tight_layout()
plt.savefig(FIG_PATH + '09_carrier_trends.png', bbox_inches='tight')
plt.show()
print("💾 Saved: 09_carrier_trends.png")

---
## 9. Airport Analysis — Which Airports Are the Worst?

Airports vary enormously in delay rates due to:
- Geographic location (weather exposure)
- Air traffic congestion
- Hub status (connecting flights amplify delays)
- Runway/gate capacity

With 420 airports, this is a **high-cardinality categorical** feature that needs careful handling.

In [ ]:
# ── 9.1 Airport delay statistics ─────────────────────────────
airport_stats = (
    df_clean
    .groupby(['airport', 'airport_name'])
    .agg(
        mean_delay_rate = ('delay_rate', 'mean'),
        std_delay_rate  = ('delay_rate', 'std'),
        total_flights   = ('arr_flights', 'sum'),
        n_records       = ('delay_rate', 'count')
    )
    .reset_index()
    .sort_values('mean_delay_rate', ascending=False)
)

# Only consider airports with enough data (at least 24 months of records)
airport_stats = airport_stats[airport_stats['n_records'] >= 24]

print(f"Airports with >= 24 months of data: {len(airport_stats)}")
print(f"\n🚨 Top 15 worst delay airports:")
print(airport_stats[['airport','airport_name','mean_delay_rate','total_flights']].head(15).to_string(index=False))

In [ ]:
# ── 9.2 Top/bottom 15 airports — side-by-side ───────────────
worst15 = airport_stats.head(15)
best15  = airport_stats.tail(15).iloc[::-1]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Worst airports
axes[0].barh(worst15['airport'], worst15['mean_delay_rate'],
             color='tomato', edgecolor='white')
axes[0].xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
axes[0].set_title('15 Worst Airports\n(mean delay rate)')
axes[0].invert_yaxis()
axes[0].set_xlabel('Mean delay rate')

# Best airports
axes[1].barh(best15['airport'], best15['mean_delay_rate'],
             color='mediumseagreen', edgecolor='white')
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
axes[1].set_title('15 Best Airports\n(mean delay rate)')
axes[1].invert_yaxis()
axes[1].set_xlabel('Mean delay rate')

plt.suptitle('Airport Delay Rate Rankings (min 24 months of data)', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_PATH + '10_airport_ranking.png', bbox_inches='tight')
plt.show()
print("💾 Saved: 10_airport_ranking.png")

In [ ]:
# ── 9.3 Volume vs delay rate scatter ─────────────────────────
# Do busier airports have higher or lower delay rates?
# This tests the "congestion hypothesis"

fig, ax = plt.subplots(figsize=(10, 6))

sc = ax.scatter(
    airport_stats['total_flights'],
    airport_stats['mean_delay_rate'],
    c=airport_stats['mean_delay_rate'],
    cmap='RdYlGn_r',
    alpha=0.6, s=30, edgecolors='none'
)
plt.colorbar(sc, ax=ax, label='Mean delay rate')

# Label the most extreme airports
for _, row in airport_stats.nlargest(5, 'mean_delay_rate').iterrows():
    ax.annotate(row['airport'], (row['total_flights'], row['mean_delay_rate']),
                fontsize=8, xytext=(5, 2), textcoords='offset points')

# Correlation line
corr = airport_stats[['total_flights','mean_delay_rate']].corr().iloc[0,1]
ax.set_xscale('log')
ax.set_xlabel('Total flights (log scale)')
ax.set_ylabel('Mean delay rate')
ax.set_title(f'Airport Volume vs Delay Rate\nPearson r = {corr:.3f}')
plt.tight_layout()
plt.savefig(FIG_PATH + '11_airport_volume_vs_delay.png', bbox_inches='tight')
plt.show()
print("💾 Saved: 11_airport_volume_vs_delay.png")

---
## 10. Bivariate Analysis — Feature vs Target Relationships

We now ask: **which input features actually correlate with the delay rate?**

This is the bridge between EDA and feature engineering — high-correlation features are  
likely to be important predictors. Low-correlation features may still be useful in  
combination (non-linear relationships that a DNN can capture).

We examine:
- Correlation of numeric features with `delay_rate`
- Scatter plots with regression lines
- Relationship between `arr_flights` volume and delay rate

In [ ]:
# ── 10.1 Pearson correlation with target ──────────────────────
# We include safe (non-leaky) numeric features only
safe_numeric = ['arr_flights', 'arr_cancelled', 'arr_diverted', 'year', 'month']

correlations = (
    df_clean[safe_numeric + ['delay_rate']]
    .corr()['delay_rate']
    .drop('delay_rate')
    .sort_values(key=abs, ascending=False)
)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['tomato' if v < 0 else 'steelblue' for v in correlations.values]
ax.barh(correlations.index, correlations.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Pearson correlation with delay_rate')
ax.set_title('Feature Correlation with Target Variable')
plt.tight_layout()
plt.savefig(FIG_PATH + '12_feature_correlations.png', bbox_inches='tight')
plt.show()

print("Correlations with delay_rate:")
print(correlations.to_string())

In [ ]:
# ── 10.2 Scatter matrix — safe numeric features vs target ─────
from pandas.plotting import scatter_matrix

cols_to_plot = ['arr_flights', 'arr_cancelled', 'arr_diverted', 'delay_rate']
sample = df_clean[cols_to_plot].sample(3000, random_state=42)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for i, col in enumerate(cols_to_plot[:-1]):
    row, c = divmod(i, 2)
    axes[row, c].scatter(
        np.log1p(sample[col]),
        sample['delay_rate'],
        alpha=0.15, s=8, color='steelblue'
    )
    # Regression line
    from numpy.polynomial.polynomial import polyfit
    x = np.log1p(sample[col].values)
    y = sample['delay_rate'].values
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() > 10:
        b, m = polyfit(x[mask], y[mask], 1)
        x_line = np.linspace(x[mask].min(), x[mask].max(), 100)
        axes[row, c].plot(x_line, b + m*x_line, 'r-', lw=1.5, alpha=0.7)

    corr_val = df_clean[[col,'delay_rate']].corr().iloc[0,1]
    axes[row, c].set_xlabel(f'log1p({col})')
    axes[row, c].set_ylabel('delay_rate')
    axes[row, c].set_title(f'{col} vs delay_rate  (r={corr_val:.3f})')

# Remove unused subplot
axes[1, 1].set_visible(False)

plt.suptitle('Scatter Plots: log1p(feature) vs delay_rate (3k-sample)', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_PATH + '13_scatter_matrix.png', bbox_inches='tight')
plt.show()
print("💾 Saved: 13_scatter_matrix.png")

---
## 11. Correlation Heatmap — All Numeric Features

The full correlation heatmap reveals **multicollinearity** — highly correlated feature pairs  
where one feature adds little information on top of the other.

**Why this matters for DNNs:**  
Unlike linear models, DNNs can handle multicollinearity better — but it still wastes capacity  
and can slow training. We'll use this to guide our feature selection in notebook 02.

**What to look for:**
- Off-diagonal blocks near ±1.0 → redundant features
- The last row/column (delay_rate) → which features actually correlate with our target

In [ ]:
# ── Full correlation heatmap ──────────────────────────────────
# Include the leaky columns here for DIAGNOSTIC purposes only
# We will clearly mark them — they should NOT enter the model

all_numeric = df_clean.select_dtypes(include=np.number).columns.tolist()

corr_matrix = df_clean[all_numeric].corr()

# Mask the upper triangle (redundant — symmetric matrix)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.3,
    annot_kws={'size': 8},
    ax=ax
)
ax.set_title(
    'Correlation Heatmap — All Numeric Features\n'
    '(⚠️  delay cause columns are LEAKY and will NOT be used as model inputs)',
    fontsize=12
)
plt.tight_layout()
plt.savefig(FIG_PATH + '14_correlation_heatmap.png', bbox_inches='tight')
plt.show()
print("💾 Saved: 14_correlation_heatmap.png")

---
## 12. Key Findings & Implications for Modelling

This section summarises everything we discovered and maps each finding to a decision  
in the next notebook (`02_feature_engineering.ipynb`).

---

### 12.1 Data Quality

| Finding | Decision |
|---|---|
| ~490 rows with nulls in numeric columns (<0.2%) | Drop these rows in preprocessing |
| 4 rows with null carrier/airport codes | Drop these rows |
| No negative values in count columns | No data corruption |
| `arr_flights > 0` required for target computation | Filter before feature creation |

---

### 12.2 Target Variable (`delay_rate`)

| Finding | Decision |
|---|---|
| Right-skewed but not severely (skew ≈ 0.7) | Try with and without sqrt transform |
| Not normally distributed (Shapiro-Wilk p < 0.05) | Use MAE + RMSE as metrics, not just MSE |
| Range [0, 1], continuous | Regression task confirmed |
| Mean ≈ 19.8%, Std ≈ 11.2% | A dummy mean-predictor has MAE ≈ 0.09 — our baseline to beat |

---

### 12.3 Temporal Patterns

| Finding | Decision |
|---|---|
| Clear 20-year trend (peak 2007, COVID collapse 2020) | Temporal split: train 2003–2018, val 2019–2020, test 2021–2022 |
| Strong seasonality: June/July worst, September/October best | Encode month as **cyclic features** (sin + cos) |
| Year × Month interaction visible in heatmap | Create `year × month` interaction feature |

---

### 12.4 Feature Engineering Roadmap

Based on this EDA, here is what we will build in notebook 02:

```
1. delay_rate                  → target (already created here)
2. month_sin, month_cos        → cyclic encoding of month (removes January/December discontinuity)
3. log1p(arr_flights)          → log-transform high-skew volume feature
4. cancel_rate                 → arr_cancelled / arr_flights
5. divert_rate                 → arr_diverted / arr_flights
6. carrier_hist_delay_rate     → carrier's rolling average delay rate (trailing 12 months)
7. airport_hist_delay_rate     → airport's rolling average delay rate (trailing 12 months)
8. is_summer                   → binary flag for June, July, August
9. is_holiday_month            → binary flag for November, December
10. carrier_label_encoded      → integer encoding (29 carriers = manageable)
11. airport_label_encoded      → integer encoding (420 airports — embedding layer in DNN)
```

---

### 12.5 Leaky Features — DO NOT USE AS MODEL INPUTS

These columns are **derived from the target** and must never be used as model features:

```
carrier_ct, weather_ct, nas_ct, security_ct, late_aircraft_ct
arr_delay, carrier_delay, weather_delay, nas_delay, security_delay, late_aircraft_delay
```

They explain WHY flights were delayed — but at prediction time, we wouldn't have this information.

---

*Next: `02_feature_engineering.ipynb` — Building the full feature matrix*

---
## 13. Save Cleaned Base DataFrame

We save the cleaned base (nulls dropped, `delay_rate` computed) so notebook 02 can  
start from here without re-running the loading and cleaning logic.

In [ ]:
# ── Save the cleaned base DataFrame ──────────────────────────
OUTPUT_PATH = '../data/processed/01_eda_clean_base.parquet'

# Select only the columns we'll need downstream (drop leaky cols from interim save)
# We keep the leaky cols for now — they will be explicitly dropped in preprocessing
df_clean.to_parquet(OUTPUT_PATH, index=False)

print(f"✅ Saved cleaned base: {OUTPUT_PATH}")
print(f"   Shape : {df_clean.shape}")
print(f"   Size  : {df_clean.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print()
print("Columns saved:")
for col in df_clean.columns:
    print(f"  {col}")